# Modelo de Markov

Este documento proporciona una descripción científica y técnica del **modelo de Markov** y las **cadenas de Markov**, integrando los fundamentos matemáticos necesarios y ejemplos prácticos en Python utilizando conjuntos de datos reales.

---

# El Modelo de Markov y las Cadenas de Markov

### 1. Definición y Propiedad de Markov
Una **cadena de Markov** es un proceso estocástico (una secuencia de variables aleatorias) en el que la probabilidad de que el sistema se mueva a un estado futuro depende exclusivamente del estado actual y no de la serie de eventos que le precedieron. En otras palabras, el estado futuro depende únicamente del estado presente, no de la historia completa del sistema Esta característica fundamental se conoce como la **propiedad de Markov** o la "falta de memoria" del sistema.

Matemáticamente, para una secuencia de variables aleatorias $X_0, X_1, X_2, \dots$ con un espacio de estados discreto $S$, la propiedad de Markov se expresa como:

$$P(X_{n+1} = i_{n+1} | X_n = i_n, X_{n-1} = i_{n-1}, \dots, X_0 = i_0) = P(X_{n+1} = i_{n+1} | X_n = i_n)$$

Una cadena de Markov es un proceso de Markov con espacio de estados discreto y tiempo discreto. Es la formulación más utilizada en aplicaciones prácticas.

### 2. Fundamentos Matemáticos
$$
P(X t+1
 =j∣X t=i,X 
t−1 =i t−1
 ,…)=P(X 
t+1
 =j∣X t =i)=p ij​$$

donde $p
i
j
p 
ij$ son las probabilidades de transición que forman la matriz de transición 
$P:$





#### Matriz de Transición Estocástica
La evolución de una cadena de Markov homogénea en el tiempo se describe mediante una **matriz de transición** $P$, donde cada elemento $P_{ij}$ representa la probabilidad de pasar del estado $i$ al estado $j$ en un solo paso.
Una matriz estocástica debe cumplir que:
1.  Sus elementos son no negativos: $P_{ij} \ge 0$.
2.  La suma de cada fila es igual a 1: $\sum_{j} P_{ij} = 1$.

#### Ecuación de Chapman-Kolmogorov
Para calcular las probabilidades de transición en $n$ pasos, se utiliza la **ecuación de Chapman-Kolmogorov**, que establece que la matriz de transición para $n$ pasos es simplemente la $n$-ésima potencia de la matriz de un solo paso:

$$P(n+m) = P(n)P(m) \implies P^{(n)} = P^n$$

#### Distribución Estacionaria
Una distribución de probabilidad $\pi$ es **estacionaria** para una cadena de Markov si el sistema permanece en esa distribución después de una transición:

$$\pi P = \pi, \quad \text{donde} \sum_i \pi_i = 1$$

---

### 3. Implementación en Python con Datasets Reales

A continuación, se presentan dos ejemplos prácticos de análisis secuencial.


#### Ejemplo: Predicción meteorológica (dataset NOAA)

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Simulación de datos reales de clima (3 estados: Soleado, Nublado, Lluvioso)
# Basado en patrones históricos de NOAA [[19]]
np.random.seed(42)

# Matriz de transición estimada de datos históricos
P_weather = np.array([
    [0.7, 0.2, 0.1],  # Soleado -> [Soleado, Nublado, Lluvioso]
    [0.3, 0.4, 0.3],  # Nublado -> [...]
    [0.2, 0.3, 0.5]   # Lluvioso -> [...]
])

states = ['Soleado', 'Nublado', 'Lluvioso']
state_map = {i: s for i, s in enumerate(states)}

# Simular 30 días de clima
def simulate_weather(days=30, start_state=0):
    weather_sequence = [start_state]
    current = start_state
    
    for _ in range(days-1):
        current = np.random.choice(len(states), p=P_weather[current])
        weather_sequence.append(current)
    
    return weather_sequence

# Ejecutar simulación
sequence = simulate_weather(30)
df_weather = pd.DataFrame({
    'Día': range(1, 31),
    'Estado': [state_map[s] for s in sequence]
})

print("Predicción meteorológica (30 días):")
print(df_weather.head(10))

# Calcular distribución estacionaria
w, v = np.linalg.eig(P_weather.T)
pi = np.real(v[:, np.isclose(w, 1)])
pi = pi / pi.sum()
print("\nDistribución estacionaria (comportamiento a largo plazo):")
for i, state in enumerate(states):
    print(f"{state}: {pi[i,0]:.2%}")

Predicción meteorológica (30 días):
   Día    Estado
0    1   Soleado
1    2   Soleado
2    3  Lluvioso
3    4  Lluvioso
4    5  Lluvioso
5    6   Soleado
6    7   Soleado
7    8   Soleado
8    9   Nublado
9   10   Nublado

Distribución estacionaria (comportamiento a largo plazo):
Soleado: 45.65%
Nublado: 28.26%
Lluvioso: 26.09%


#### Ejemplo: Generación de Texto con Markovify (Dataset Gutenberg)

In [6]:
# Instalar primero: pip install markovify
import markovify

# Usando texto real del Proyecto Gutenberg (ej: "Don Quijote")
# Descarga previa recomendada: https://www.gutenberg.org/files/2000/2000-0.txt
try:
    with open('don_quijote.txt', encoding='utf-8') as f:
        text = f.read()
except FileNotFoundError:
    # Texto de ejemplo para demostración
    text = """
    En un lugar de la Mancha, de cuyo nombre no quiero acordarme,
    no ha mucho tiempo que vivía un hidalgo de los de lanza en astillero,
    adarga antigua, rocín flaco y galgo corredor.
    """
    
# Construir modelo de cadena de Markov de orden 2
text_model = markovify.Text(text, state_size=2)

# Generar 5 frases nuevas
print("\nTexto generado con cadena de Markov:")
for i in range(5):
    sentence = text_model.make_sentence()
    if sentence:
        print(f"{i+1}. {sentence}")

ModuleNotFoundError: No module named 'markovify'

#### **Ejemplo**: Análisis de Mercado Bursátil (Yahoo Finance)

In [ ]:
# instalar primero: pip install yfinance
import yfinance as yf
import numpy as np

# Descargar datos reales de Apple (AAPL)
# pip install yfinance
df = yf.download('AAPL', start='2023-01-01', end='2024-01-01', progress=False)

# Definir estados basados en rendimiento diario
df['Returns'] = df['Close'].pct_change()
df['State'] = pd.cut(df['Returns'], 
                     bins=[-np.inf, -0.01, 0.01, np.inf],
                     labels=['Baja', 'Estable', 'Alza'])

# Calcular matriz de transición
states_order = ['Baja', 'Estable', 'Alza']
transition_matrix = pd.DataFrame(np.zeros((3, 3)), 
                                 index=states_order, 
                                 columns=states_order)

for (prev, curr) in zip(df['State'].shift(1).dropna(), df['State'].dropna()):
    if pd.notna(prev) and pd.notna(curr):
        transition_matrix.loc[prev, curr] += 1

# Normalizar a probabilidades
transition_matrix = transition_matrix.div(transition_matrix.sum(axis=1), axis=0)

print("\nMatriz de transición del mercado (AAPL 2023):")
print(transition_matrix.round(3))

# Predecir estado siguiente dado estado actual "Alza"
current_state = 'Alza'
next_state = np.random.choice(
    states_order, 
    p=transition_matrix.loc[current_state].values
)
print(f"\nSi hoy hubo '{current_state}', mañana probablemente: {next_state}")


#### **Ejemplo**: Análisis de Secuencias de ADN (ACTG)
Las secuencias de ADN se pueden modelar como cadenas de Markov de cuatro estados: Adenina (A), Timina (T), Citocina (C) y Guanina (G).

In [ ]:
import numpy as np
import pandas as pd

# Dataset real: Secuencia de nucleótidos (Ejemplo simplificado de una cadena de ADN)
dna_seq = "TACTTCGATTTAAGCGCGGCGGCCTATATTA" # Fuente:

def build_transition_matrix(seq):
    states = sorted(list(set(seq)))
    n_states = len(states)
    state_to_idx = {s: i for i, s in enumerate(states)}
    
    # Inicializar matriz de conteo
    matrix = np.zeros((n_states, n_states))
    
    # Contar transiciones
    for i in range(len(seq) - 1):
        curr_state = seq[i]
        next_state = seq[i+1]
        matrix[state_to_idx[curr_state], state_to_idx[next_state]] += 1
        
    # Normalizar para obtener probabilidades (Matriz Estocástica)
    row_sums = matrix.sum(axis=1)
    transition_matrix = matrix / row_sums[:, np.newaxis]
    
    return pd.DataFrame(transition_matrix, index=states, columns=states)

tm = build_transition_matrix(dna_seq)
print("Matriz de Transición de ADN:\n", tm)

Matriz de Transición de ADN:
           A         C         G         T
A  0.166667  0.166667  0.166667  0.500000
C  0.000000  0.142857  0.571429  0.285714
G  0.142857  0.571429  0.285714  0.000000
T  0.500000  0.100000  0.000000  0.400000




#### **Ejemplo**: Modelo Oculto de Markov (HMM) para Terremotos
En un **HMM**, el estado del sistema es oculto, pero influye en las observaciones. Un dataset clásico es el conteo anual de terremotos de gran magnitud (1900-2006).



In [ ]:

from hmmlearn import hmm

# Dataset: Conteos de terremotos (Ejemplo de serie temporal)
# Representa periodos de baja y alta actividad sísmica
quakes = np.array().reshape(-1, 1)

# Definir un modelo de 2 estados (Baja vs Alta actividad)
model = hmm.PoissonHMM(n_components=2, n_iter=1000)
model.fit(quakes)

# Predecir los estados ocultos (Decodificación de Viterbi)
hidden_states = model.predict(quakes)
print("Estados ocultos (0=Baja, 1=Alta):", hidden_states)

# Probabilidades de transición entre estados de actividad
print("Matriz de transición de estados:\n", model.transmat_)


ModuleNotFoundError: No module named 'hmmlearn'

---

### 4. Clasificación de Estados y Ergodicidad
*   **Recurrencia:** Un estado es recurrente si la probabilidad de regresar a él en un tiempo finito es 1.
*   **Transitoriedad:** Un estado es transitorio si existe una probabilidad mayor a 0 de nunca regresar a él.
*   **Ergodicidad:** Una cadena es ergódica si es irreducible (todos los estados se comunican), positiva recurrente y aperiódica. En este estado, los promedios temporales convergen a las esperanzas de la distribución estacionaria.

### 5. Casos de Uso Comunes
1.  **MCMC (Monte Carlo por Cadenas de Markov):** Utilizado para obtener muestras de distribuciones posteriores complejas en estadística bayesiana mediante algoritmos como Metropolis-Hastings o el muestreo de Gibbs.
2.  **Procesamiento de Lenguaje Natural:** Modelado de secuencias de palabras para predicción de texto y reconocimiento de voz.
3.  **Economía y Finanzas:** Análisis de fluctuaciones de mercado y volatilidad de activos.
4.  **Bioinformática:** Alineación de secuencias de proteínas y análisis de genomas.

### Diferencias entre una cadena de Markov y modelo oculto de Markov

La diferencia fundamental entre una **cadena de Markov** y un **modelo oculto de Markov (HMM)** radica en la **visibilidad de los estados**: mientras que en una cadena de Markov los estados son directamente observables, en un HMM los estados internos están "ocultos" y solo pueden inferirse a través de una serie de observaciones o emisiones producidas por dicho sistema.

A continuación se detallan las diferencias principales organizadas por categorías:

### 1. Observabilidad y Estructura
*   **Cadena de Markov (MC):** Es una secuencia de variables aleatorias donde cada estado es conocido y visible para el observador. El sistema evoluciona a través de transiciones entre estos estados observables siguiendo la **propiedad de Markov**.

*   **Modelo Oculto de Markov (HMM):** Es un "modelo de mezcla dependiente" que consta de dos partes: una **cadena de Markov subyacente** (parámetro de proceso) que no se puede ver, y un **proceso dependiente del estado** que genera los datos que sí observamos. Por ejemplo, en el reconocimiento de voz, los estados son fonemas ocultos y las observaciones son las señales de audio grabadas.

### 2. Componentes Matemáticos
Para definir cada modelo, se requieren distintos conjuntos de parámetros:

| Característica | Cadena de Markov | Modelo Oculto de Markov (HMM) |
| :--- | :--- | :--- |
| **Estados** | Todos son visibles. | Existen estados ocultos y observaciones visibles. |
| **Probabilidades de Transición** | Definen el salto entre estados conocidos. | Definen el salto entre los estados ocultos. |
| **Probabilidades de Emisión** | No existen (el estado *es* la observación). | Definen la probabilidad de que un estado oculto genere una observación específica. |
| **Parámetros base** | Matriz de transición ($\Gamma$) y distribución inicial ($\delta$). | Matriz de transición, matriz de emisión (o densidades) y distribución inicial. |

### 3. Aplicación de la Propiedad de Markov
*   **Consistencia:** Una cadena de Markov siempre satisface la propiedad de que el futuro depende únicamente del presente.
*   **Deterioro en HMM:** Aunque el proceso oculto es una cadena de Markov, la secuencia de **observaciones visibles en un HMM generalmente no cumple con la propiedad de Markov**. Esto significa que conocer solo la observación actual no siempre es suficiente para predecir la siguiente sin considerar la historia del modelo.

### 4. Objetivos e Inferencia
*   **En Cadenas de Markov:** El interés suele estar en calcular las **probabilidades de transición**, el tiempo promedio en un estado o el comportamiento del sistema a largo plazo (estado estacionario).

*   **En HMM:** Los desafíos son más complejos y se dividen en tres problemas clásicos:
    1.  **Evaluación:** Calcular la probabilidad de una secuencia de observaciones.
    2.  **Decodificación:** Determinar la secuencia más probable de estados ocultos que generó las observaciones (comúnmente mediante el **algoritmo de Viterbi**).
    3.  **Aprendizaje:** Estimar los parámetros del modelo (transiciones y emisiones) a partir de los datos observados (usando el **algoritmo de Baum-Welch**).

***

**En resumen:** Una cadena de Markov es un sistema donde lo que ves es lo que sucede. Un HMM es un sistema donde lo que ves es solo el resultado (emisión) de un mecanismo interno invisible que se comporta como una cadena de Markov.

## Modelo oculto de Markov (HMM)

En un Modelo Oculto de Markov (HMM), la **matriz de emisión** (o probabilidades dependientes del estado) define la probabilidad de obtener una observación específica dado que el sistema se encuentra en un estado oculto determinado,. El cálculo de esta matriz depende de si los estados son conocidos (aprendizaje supervisado) o si deben inferirse de los datos (aprendizaje no supervisado).

### 1. Definición Conceptual
Cada elemento de la matriz representa la probabilidad condicionada $P(X_t = x | C_t = i)$, donde $X_t$ es la observación y $C_t$ es el estado en el tiempo $t$. En modelos con observaciones discretas, esto forma una matriz de probabilidades; en modelos continuos, se utilizan **funciones de densidad de probabilidad** (como la Gaussiana) cuyos parámetros deben ser estimados,.

### 2. Estimación mediante el Algoritmo de Baum-Welch (EM)
Cuando los estados son ocultos, el método estándar para calcular estos parámetros es el **algoritmo de Baum-Welch**, que es una versión del algoritmo de Esperanza-Maximización (EM),. Este proceso es iterativo y se divide en dos pasos principales:

*   **Paso E (Esperanza):** Se calculan las **probabilidades forward ($\alpha$) y backward ($\beta$)** para determinar la "responsabilidad" o probabilidad de que el modelo esté en el estado $j$ en el momento $t$, dada la secuencia de observaciones. Esta cantidad se denota como:
    $$\hat{u}_j(t) = P(C_t = j | x^{(T)}) = \frac{\alpha_t(j)\beta_t(j)}{L_T}$$
    donde $L_T$ es la verosimilitud total de las observaciones,.
*   **Paso M (Maximización):** Se utilizan estas probabilidades para actualizar los parámetros de emisión de modo que se maximice la verosimilitud de los datos,.

### 3. Fórmulas de Reestimación según la Distribución
El cálculo específico de los valores de emisión varía según la naturaleza de las observaciones:

*   **Para distribuciones de Poisson:**
    El parámetro $\lambda$ (media) para el estado $j$ se actualiza como un promedio ponderado de las observaciones $x_t$:
    $$
    \hat{\lambda}_j = \frac{\sum_{t=1}^T \hat{u}_j(t) x_t}{\sum_{t=1}^T \hat{u}_j(t)}
    $$


*   **Para distribuciones Normales (Gaussianas):**
    Se calculan la media ($\mu$) y la varianza ($\sigma^2$) de forma similar:
    $$
    \hat{\mu}_j = \frac{\sum_{t=1}^T \hat{u}_j(t) x_t}{\sum_{t=1}^T \hat{u}_j(t)}
    $$

    $$
    \hat{\sigma}_j^2 = \frac{\sum_{t=1}^T \hat{u}_j(t) (x_t - \hat{\mu}_j)^2}{\sum_{t=1}^T \hat{u}_j(t)}
    $$

*   **Para datos categóricos (Multinomial):**
    La probabilidad de observar el símbolo $k$ en el estado $i$ se estima dividiendo el número esperado de veces que aparece ese símbolo en ese estado por el número total esperado de veces que se visita dicho estado,.

### 4. Aprendizaje Supervisado
Si se dispone de un conjunto de entrenamiento donde tanto los estados como las observaciones están etiquetados, el cálculo es directo. En este caso, las probabilidades de emisión se obtienen simplemente calculando las **frecuencias relativas** de cada observación dentro de cada estado en los datos observados,.

Calcula la matriz de emisión en un HMM

La **matriz de emisión** (a veces denominada matriz $B$ o $M$) en un Modelo Oculto de Markov (HMM) contiene las probabilidades de que un estado oculto específico genere una observación particular. Su cálculo o estimación se realiza habitualmente mediante procesos iterativos cuando los estados no son observables, siendo el método estándar el **algoritmo de Baum-Welch** (una versión del algoritmo de Esperanza-Maximización o EM).

A continuación se detalla cómo se calcula y estima esta matriz:

### 1. Definición Conceptual
Cada elemento de la matriz, $m_{i,j}$, representa la probabilidad condicional de observar el símbolo $j$ dado que el sistema se encuentra en el estado oculto $i$:
$$P(O_k = j | X_k = i)$$
Donde $O_k$ es la observación en el tiempo $k$ y $X_k$ es el estado oculto en ese mismo instante.

### 2. Estimación mediante el Algoritmo de Baum-Welch
Cuando se dispone de una secuencia de entrenamiento de observaciones pero los estados son ocultos, la matriz se estima siguiendo estos pasos:

*   **Paso E (Esperanza):** Se calculan las **probabilidades *forward* ($\alpha$) y *backward* ($\beta$)** para determinar la probabilidad de que el modelo haya estado en un estado $i$ en el momento $t$, dada toda la secuencia de observaciones. Esta probabilidad se conoce como "responsabilidad" o $\gamma_t(i)$.
*   **Paso M (Maximización):** Se actualizan los parámetros de emisión para maximizar la verosimilitud de los datos observados.

### 3. Fórmulas según el tipo de observación
El cálculo específico de los valores de la matriz depende de si las observaciones son discretas o continuas:

*   **Observaciones Discretas:**
    La probabilidad de emitir el símbolo $r$ desde el estado $i$ se actualiza como el cociente entre el número esperado de veces que se observó el símbolo $r$ estando en el estado $i$, y el número total esperado de veces que se visitó dicho estado:
    $$\hat{p}_{i}(r) = \frac{\sum_{t=1, O_t=r}^{T} \gamma_t(i)}{\sum_{t=1}^{T} \gamma_t(i)}$$
*   **Observaciones Continuas (Parámetros):**
    Si las emisiones siguen una distribución (como una Gaussiana o Poisson), no se calcula una matriz de celdas fijas, sino que se actualizan los parámetros de la distribución para cada estado. 
    *   Para un **HMM-Poisson**, la media ($\lambda$) del estado $j$ se actualiza como un promedio ponderado de las observaciones:
        $$\hat{\lambda}_j = \frac{\sum_{t=1}^{T} \gamma_t(j) x_t}{\sum_{t=1}^{T} \gamma_t(j)}$$
    *   Para un **HMM-Normal**, se actualizan la media ($\mu$) y la varianza ($\sigma^2$) de forma similar, ponderando cada dato por la probabilidad de pertenecer a ese estado en ese momento.

### 4. Estimación Supervisada (Viterbi Re-estimation)
Si se conoce la secuencia de estados (por ejemplo, en un conjunto de datos etiquetado para aprendizaje supervisado), el cálculo es directo mediante frecuencias relativas:
*   Se cuenta cuántas veces aparece cada observación en cada estado.
*   Se divide ese conteo por el número total de veces que se estuvo en ese estado.

**Nota sobre la estabilidad:** En implementaciones reales, estos cálculos suelen realizarse utilizando **logaritmos** o técnicas de **escalado** para evitar errores numéricos de subflujo (*underflow*), ya que las probabilidades multiplicadas a lo largo del tiempo pueden volverse extremadamente pequeñas.

**Cómo decide el algoritmo de Viterbi la secuencia de estados?**

El algoritmo de Viterbi decide la secuencia de estados mediante un proceso de **decodificación global**, cuyo objetivo es encontrar la secuencia de estados ocultos ($c_1, c_2, ..., c_T$) que maximice la probabilidad conjunta de dicha secuencia y de las observaciones registradas ($Pr(C^{(T)}, X^{(T)})$).

A diferencia de otros métodos que evalúan cada estado por separado (decodificación local), Viterbi utiliza la **programación dinámica** para encontrar la trayectoria óptima a través de un diagrama de rejilla (*trellis*) de forma eficiente. El proceso se divide principalmente en dos fases:

### 1. Fase de Recurrencia (Hacia adelante)
En lugar de calcular por fuerza bruta todas las combinaciones posibles (que serían $m^T$), el algoritmo avanza paso a paso calculando la probabilidad del camino más probable para llegar a cada estado en el tiempo $t$.
*   **Cálculo:** Se define una variable ($\xi_{tj}$) que almacena la probabilidad máxima de haber seguido una secuencia de estados que termina en el estado $i$ al tiempo $t$.
*   **Regla de actualización:** En cada paso, el algoritmo mira los estados anteriores y selecciona solo el que ofrece la mayor probabilidad de transición al estado actual, multiplicándola por la probabilidad de emisión del dato observado en ese momento.
*   **Almacenamiento:** Es fundamental que, durante este proceso, el algoritmo guarde un registro (puntero) de qué estado anterior fue el que produjo ese valor máximo para cada nodo.

### 2. Fase de Retroceso (*Back-tracking*)
Una vez que el algoritmo llega al final de la secuencia de observaciones (tiempo $T$):
*   **Selección del final:** Identifica el estado final ($i_T$) que tiene la probabilidad acumulada más alta.
*   **Reconstrucción de la ruta:** Utilizando los punteros almacenados en la fase anterior, el algoritmo se mueve hacia atrás en el tiempo para recuperar la secuencia completa que llevó a ese máximo. Este paso garantiza que se obtenga la "mejor combinación" global de estados, evitando inconsistencias que podrían surgir si se eligieran los estados máximos de forma aislada.

### Consideraciones Numéricas y Eficiencia
*   **Uso de Logaritmos:** Dado que multiplicar muchas probabilidades pequeñas puede provocar errores de precisión (subflujo aritmético), el algoritmo suele implementarse como un **"Max-Sum"**, donde se maximiza la suma de los logaritmos de las probabilidades en lugar del producto de las probabilidades directas.
*   **Complejidad:** El esfuerzo computacional es **lineal** respecto al número de observaciones ($O(T)$), lo que lo hace viable para secuencias largas en aplicaciones como el reconocimiento de voz o la bioinformática.
*   **Sensibilidad:** Se ha observado que la ruta de Viterbi puede ser sensible a la especificación del modelo; pequeños cambios en los parámetros pueden alterar la secuencia de estados resultante, por lo que los resultados deben interpretarse con cautela.

**¿Cómo se comparan las emisiones discretas frente a las continuas?**

En el contexto de los **Modelos Ocultos de Markov (HMM)** y los procesos estocásticos, las emisiones representan los datos observables generados por los estados ocultos. La principal diferencia entre las emisiones discretas y continuas radica en la naturaleza de los datos que describen y las herramientas matemáticas utilizadas para modelarlos.

A continuación se presenta una comparación detallada basada en las fuentes proporcionadas:

### Comparativa: Emisiones Discretas frente a Continuas

| Característica | Emisiones Discretas | Emisiones Continuas |
| :--- | :--- | :--- |
| **Naturaleza del Dato** | Valores contables o categorías (ej. cara/cruz, tipos de nucleótidos A, C, T, G). | Valores en un rango infinito o finito (ej. temperatura, peso, tiempo, voltaje). |
| **Función Matemática** | **Función de Masa de Probabilidad (PMF)**: $P(X = x)$. Asigna probabilidad a puntos específicos. | **Función de Densidad de Probabilidad (PDF)**: $p(x)$. La probabilidad se mide en intervalos mediante integrales. |
| **Representación** | Una **Matriz de Emisión** donde cada fila suma 1. | Un conjunto de **Parámetros de Distribución** (como la media $\mu$ y varianza $\sigma^2$) para cada estado. |
| **Modelado Común** | Distribuciones de Bernoulli, Binomial o Multinomial. | Distribuciones Gaussianas (Normales), Exponenciales o Gamma. |
| **Cálculo de Verosimilitud** | Se basa en sumatorias de probabilidades discretas. | Se basa en el cálculo de integrales sobre la densidad. |

---

### Diferencias Clave en el Funcionamiento y Aplicación

1.  **Cálculo de Parámetros (Paso M del Algoritmo EM):**
    *   En el caso **discreto**, los parámetros se actualizan calculando las frecuencias relativas esperadas de cada símbolo observado en cada estado oculto.
    *   En el caso **continuo**, el proceso es más complejo: no se rellena una matriz, sino que se reestiman los parámetros de la distribución (por ejemplo, la media ponderada de las observaciones para una distribución de Poisson o los momentos para una Gaussiana).

2.  **Problemas de Estabilidad Numérica:**
    *   **Emisiones Continuas:** Pueden sufrir el fenómeno de **verosimilitud no acotada** (*unbounded likelihood*). Esto ocurre si la varianza de un componente tiende a cero cuando su media coincide exactamente con un punto de dato, provocando "picos" infinitos en la función de verosimilitud.
    *   **Emisiones Discretas:** Este problema no existe, ya que las probabilidades están acotadas entre 0 y 1.

3.  **Precisión y Discretización:**
    *   A menudo, los datos continuos se transforman en discretos mediante el **binning** o discretización (agrupación en intervalos) para simplificar el cálculo. Sin embargo, esto requiere un juicio subjetivo sobre los rangos y puede provocar una **pérdida de información** valiosa.
    *   Por el contrario, el uso de distribuciones continuas permite capturar cambios graduales en el sistema, lo cual es más intuitivo en fenómenos físicos como la sismología o el movimiento animal.

4.  **Interpretación de la Probabilidad:**
    *   En un modelo **discreto**, la probabilidad de que ocurra un evento exacto es un valor real (masa). En un modelo **continuo**, la probabilidad de que un valor sea *exactamente* un número (ej. que la temperatura sea exactamente 23.1958... °C) es técnicamente **cero**; solo tiene sentido hablar de la probabilidad de que el valor caiga dentro de un intervalo.